# VoxIntel — 12: Consolidate Reports and Continue the Research

## Goal

This notebook creates one clean research state after the debugging phase.

It uses the work from:

- Notebook 09 (original taxonomy attempt)
- Notebook 10 (diagnosis and debugging)
- Notebook 11 (fixed taxonomy pipeline)

and then:

1. backs up the broken Notebook 09 canonical taxonomy reports
2. deletes those broken canonical files from `reports/`
3. promotes the corrected Notebook 11 outputs into the canonical report names
4. writes a canonical manifest and continuation plan

The purpose is simple:

- stop carrying broken and fixed report versions in parallel as if they were equally trustworthy
- make the `reports/` directory scientifically cleaner
- resume research from a stable canonical state

## Important safety note

This notebook does not blindly wipe the reports directory.
It only targets the taxonomy files that Notebook 10 showed were broken.
Everything is backed up first into a timestamped archive directory.


## Cell 0 — Make sure `src` is importable


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: c:\Users\ACER\OneDrive\Desktop\VoxIntel


## Cell 1 — Imports


In [2]:
import json
import shutil
from datetime import datetime

import pandas as pd
from IPython.display import display


## Cell 2 — Define file groups

We separate files into three conceptual groups.

### Broken canonical taxonomy outputs from Notebook 09
These were the files shown by Notebook 10 to be untrustworthy.

### Fixed taxonomy outputs from Notebook 11
These are the repaired versions we want to promote.

### Stable trusted files from Notebooks 07, 08, and 10
These remain part of the trusted project state and are not deleted.


In [3]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
ARCHIVE_DIR = REPORTS_DIR / f"archive_broken_taxonomy_{RUN_STAMP}"

BROKEN_CANONICAL_FILES = {
    "error_taxonomy_dataset.csv": REPORTS_DIR / "error_taxonomy_dataset.csv",
    "error_type_distribution.csv": REPORTS_DIR / "error_type_distribution.csv",
    "error_impact_matrix.csv": REPORTS_DIR / "error_impact_matrix.csv",
    "intent_fragility.csv": REPORTS_DIR / "intent_fragility.csv",
    "failure_gallery.csv": REPORTS_DIR / "failure_gallery.csv",
    "recovery_summary.csv": REPORTS_DIR / "recovery_summary.csv",
}

FIXED_FILES = {
    "error_taxonomy_dataset_fixed.csv": REPORTS_DIR / "error_taxonomy_dataset_fixed.csv",
    "error_type_distribution_fixed.csv": REPORTS_DIR / "error_type_distribution_fixed.csv",
    "error_impact_matrix_fixed.csv": REPORTS_DIR / "error_impact_matrix_fixed.csv",
    "intent_fragility_fixed.csv": REPORTS_DIR / "intent_fragility_fixed.csv",
    "failure_gallery_fixed.csv": REPORTS_DIR / "failure_gallery_fixed.csv",
    "recovery_summary_fixed.csv": REPORTS_DIR / "recovery_summary_fixed.csv",
    "taxonomy_fix_summary.json": REPORTS_DIR / "taxonomy_fix_summary.json",
}

PROMOTION_MAP = {
    REPORTS_DIR / "error_taxonomy_dataset_fixed.csv": REPORTS_DIR / "error_taxonomy_dataset.csv",
    REPORTS_DIR / "error_type_distribution_fixed.csv": REPORTS_DIR / "error_type_distribution.csv",
    REPORTS_DIR / "error_impact_matrix_fixed.csv": REPORTS_DIR / "error_impact_matrix.csv",
    REPORTS_DIR / "intent_fragility_fixed.csv": REPORTS_DIR / "intent_fragility.csv",
    REPORTS_DIR / "failure_gallery_fixed.csv": REPORTS_DIR / "failure_gallery.csv",
    REPORTS_DIR / "recovery_summary_fixed.csv": REPORTS_DIR / "recovery_summary.csv",
}

STABLE_TRUSTED_FILES = {
    "metric_comparison.csv": REPORTS_DIR / "metric_comparison.csv",
    "confidence_comparison.csv": REPORTS_DIR / "confidence_comparison.csv",
    "error_propagation_results.csv": REPORTS_DIR / "error_propagation_results.csv",
    "debugging_summary_07_08_09.json": REPORTS_DIR / "debugging_summary_07_08_09.json",
    "debugging_verdict_07_08_09.csv": REPORTS_DIR / "debugging_verdict_07_08_09.csv",
}


## Cell 3 — Show current availability before cleanup


In [4]:
rows = []
for name, path in BROKEN_CANONICAL_FILES.items():
    rows.append({"group": "broken_canonical", "file": name, "exists": path.exists(), "path": str(path)})
for name, path in FIXED_FILES.items():
    rows.append({"group": "fixed_outputs", "file": name, "exists": path.exists(), "path": str(path)})
for name, path in STABLE_TRUSTED_FILES.items():
    rows.append({"group": "stable_trusted", "file": name, "exists": path.exists(), "path": str(path)})

availability_df = pd.DataFrame(rows)
display(availability_df)


,group,file,exists,path
0,broken_canonical,error_taxonomy_dataset.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
1,broken_canonical,error_type_distribution.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
2,broken_canonical,error_impact_matrix.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
3,broken_canonical,intent_fragility.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
4,broken_canonical,failure_gallery.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
5,broken_canonical,recovery_summary.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
6,fixed_outputs,error_taxonomy_dataset_fixed.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
7,fixed_outputs,error_type_distribution_fixed.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
8,fixed_outputs,error_impact_matrix_fixed.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
9,fixed_outputs,intent_fragility_fixed.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...


## Cell 4 — Safety gate: require the fixed Notebook 11 outputs


In [5]:
missing_fixed = [name for name, path in FIXED_FILES.items() if not path.exists() and name != "taxonomy_fix_summary.json"]
if missing_fixed:
    raise FileNotFoundError(
        "Cannot consolidate reports yet. Missing fixed Notebook 11 outputs: " + ", ".join(missing_fixed)
    )

print("All required fixed Notebook 11 outputs are present.")


All required fixed Notebook 11 outputs are present.


## Cell 5 — Preview the fixed outputs before promotion


In [6]:
preview_files = [
    REPORTS_DIR / "error_type_distribution_fixed.csv",
    REPORTS_DIR / "error_impact_matrix_fixed.csv",
    REPORTS_DIR / "intent_fragility_fixed.csv",
    REPORTS_DIR / "recovery_summary_fixed.csv",
]

for path in preview_files:
    if path.exists():
        print()
        print("=" * 80)
        print(path.name)
        display(pd.read_csv(path).head(15))



error_type_distribution_fixed.csv


,error_type,baseline,finetuned,delta_finetuned_minus_baseline
0,deletion,19,59,40
1,entity_error,43,46,3
2,no_error,565,1843,1278
3,number_error,488,253,-235
4,proper_noun_error,7573,6487,-1086



error_impact_matrix_fixed.csv


,system,error_type,samples,intent_failure_rate,low_impact_rate,mean_edits,mean_overlap
0,baseline,deletion,19,0.210526,0.789474,1.105263,0.787425
1,baseline,entity_error,43,0.093023,0.906977,1.325581,0.722733
2,baseline,no_error,565,0.203540,0.000000,0.000000,1.000000
3,baseline,number_error,488,0.682377,0.317623,6.598361,0.229696
4,baseline,proper_noun_error,7573,0.645451,0.354549,4.054008,0.302066
5,finetuned,deletion,59,0.084746,0.915254,1.084746,0.796427
6,finetuned,entity_error,46,0.043478,0.956522,1.152174,0.769953
7,finetuned,no_error,1843,0.158980,0.000000,0.000000,1.000000
8,finetuned,number_error,253,0.288538,0.711462,3.861660,0.488153
9,finetuned,proper_noun_error,6487,0.298289,0.701711,2.310775,0.529471



intent_fragility_fixed.csv


,intent,n_samples,baseline_failure_rate,finetuned_failure_rate,baseline_mean_edits,finetuned_mean_edits,recovery_rate
0,cooking_query,14,1.0,1.000000,1.214286,1.357143,0.000000
1,cleaning,8,1.0,1.000000,5.125000,1.625000,0.000000
2,general_greet,17,1.0,1.000000,2.058824,0.823529,0.000000
3,factoid,14,1.0,1.000000,2.714286,1.285714,0.000000
4,hue_lightup,1,1.0,1.000000,4.000000,1.000000,0.000000
5,podcasts,6,1.0,1.000000,2.833333,0.833333,0.000000
6,post,2,1.0,1.000000,3.500000,2.000000,0.000000
7,music_dislikeness,5,1.0,1.000000,2.600000,1.400000,0.000000
8,greet,2,1.0,1.000000,0.500000,0.000000,0.000000
9,hue_lightoff,9,1.0,1.000000,3.444444,1.111111,0.000000



recovery_summary_fixed.csv


,recovery_status,samples,rate
0,recovered,3430,0.394797
1,always_correct,2950,0.339549
2,still_broken,1914,0.220304
3,regressed,394,0.045350


## Cell 6 — Archive the broken Notebook 09 canonical files

We preserve the old files first.
That keeps the debugging trail intact and makes the cleanup reversible.


In [7]:
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
archived_rows = []

for name, path in BROKEN_CANONICAL_FILES.items():
    if path.exists():
        archive_path = ARCHIVE_DIR / name
        shutil.copy2(path, archive_path)
        archived_rows.append({"file": name, "archived_to": str(archive_path), "status": "archived"})
    else:
        archived_rows.append({"file": name, "archived_to": None, "status": "missing_before_archive"})

archive_manifest_df = pd.DataFrame(archived_rows)
display(archive_manifest_df)


,file,archived_to,status
0,error_taxonomy_dataset.csv,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...,archived
1,error_type_distribution.csv,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...,archived
2,error_impact_matrix.csv,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...,archived
3,intent_fragility.csv,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...,archived
4,failure_gallery.csv,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...,archived
5,recovery_summary.csv,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...,archived


## Cell 7 — Delete the broken canonical taxonomy files from active reports


In [8]:
delete_rows = []
for name, path in BROKEN_CANONICAL_FILES.items():
    if path.exists():
        path.unlink()
        delete_rows.append({"file": name, "deleted": True})
    else:
        delete_rows.append({"file": name, "deleted": False})

delete_manifest_df = pd.DataFrame(delete_rows)
display(delete_manifest_df)


,file,deleted
0,error_taxonomy_dataset.csv,True
1,error_type_distribution.csv,True
2,error_impact_matrix.csv,True
3,intent_fragility.csv,True
4,failure_gallery.csv,True
5,recovery_summary.csv,True


## Cell 8 — Promote the fixed Notebook 11 outputs into canonical report names

This is the key transition point.
After this step, future notebooks can once again read the canonical taxonomy filenames.


In [9]:
promotion_rows = []
for src, dst in PROMOTION_MAP.items():
    if not src.exists():
        raise FileNotFoundError(f"Expected fixed file missing: {src}")
    shutil.copy2(src, dst)
    promotion_rows.append({"source": src.name, "promoted_to": dst.name, "status": "copied"})

promotion_manifest_df = pd.DataFrame(promotion_rows)
display(promotion_manifest_df)


,source,promoted_to,status
0,error_taxonomy_dataset_fixed.csv,error_taxonomy_dataset.csv,copied
1,error_type_distribution_fixed.csv,error_type_distribution.csv,copied
2,error_impact_matrix_fixed.csv,error_impact_matrix.csv,copied
3,intent_fragility_fixed.csv,intent_fragility.csv,copied
4,failure_gallery_fixed.csv,failure_gallery.csv,copied
5,recovery_summary_fixed.csv,recovery_summary.csv,copied


## Cell 9 — Verify the promoted canonical files now exist


In [10]:
verification_rows = []
for name, path in BROKEN_CANONICAL_FILES.items():
    verification_rows.append({
        "file": name,
        "exists_after_promotion": path.exists(),
        "path": str(path),
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)


,file,exists_after_promotion,path
0,error_taxonomy_dataset.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
1,error_type_distribution.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
2,error_impact_matrix.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
3,intent_fragility.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
4,failure_gallery.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...
5,recovery_summary.csv,True,c:\Users\ACER\OneDrive\Desktop\VoxIntel\report...


## Cell 10 — Load the new canonical report set


In [11]:
canonical_error_taxonomy_df = pd.read_csv(REPORTS_DIR / "error_taxonomy_dataset.csv")
canonical_error_type_distribution_df = pd.read_csv(REPORTS_DIR / "error_type_distribution.csv")
canonical_error_impact_matrix_df = pd.read_csv(REPORTS_DIR / "error_impact_matrix.csv")
canonical_intent_fragility_df = pd.read_csv(REPORTS_DIR / "intent_fragility.csv")
canonical_failure_gallery_df = pd.read_csv(REPORTS_DIR / "failure_gallery.csv")
canonical_recovery_summary_df = pd.read_csv(REPORTS_DIR / "recovery_summary.csv")

print("Canonical files loaded successfully.")
print(canonical_error_taxonomy_df.shape)
print(canonical_error_type_distribution_df.shape)
print(canonical_error_impact_matrix_df.shape)
print(canonical_intent_fragility_df.shape)
print(canonical_failure_gallery_df.shape)
print(canonical_recovery_summary_df.shape)


Canonical files loaded successfully.
(8688, 41)
(5, 4)
(10, 7)
(70, 7)
(5738, 16)
(4, 3)


## Cell 11 — Theory: active report state vs archival report state

A clean research project benefits from separating two different histories.

### Archival history
This preserves what happened, including broken experiments and intermediate outputs.

### Active canonical state
This represents what the project currently trusts and builds upon.

Notebook 12 creates that separation explicitly.
That improves reproducibility and reduces the chance that a later notebook accidentally reads a broken artifact.


## Cell 12 — Summarize the corrected canonical taxonomy outputs


In [12]:
display(canonical_error_type_distribution_df)
display(canonical_error_impact_matrix_df.head(20))
display(canonical_intent_fragility_df.head(20))
display(canonical_recovery_summary_df)


,error_type,baseline,finetuned,delta_finetuned_minus_baseline
0,deletion,19,59,40
1,entity_error,43,46,3
2,no_error,565,1843,1278
3,number_error,488,253,-235
4,proper_noun_error,7573,6487,-1086


,system,error_type,samples,intent_failure_rate,low_impact_rate,mean_edits,mean_overlap
0,baseline,deletion,19,0.210526,0.789474,1.105263,0.787425
1,baseline,entity_error,43,0.093023,0.906977,1.325581,0.722733
2,baseline,no_error,565,0.203540,0.000000,0.000000,1.000000
3,baseline,number_error,488,0.682377,0.317623,6.598361,0.229696
4,baseline,proper_noun_error,7573,0.645451,0.354549,4.054008,0.302066
5,finetuned,deletion,59,0.084746,0.915254,1.084746,0.796427
6,finetuned,entity_error,46,0.043478,0.956522,1.152174,0.769953
7,finetuned,no_error,1843,0.158980,0.000000,0.000000,1.000000
8,finetuned,number_error,253,0.288538,0.711462,3.861660,0.488153
9,finetuned,proper_noun_error,6487,0.298289,0.701711,2.310775,0.529471


,intent,n_samples,baseline_failure_rate,finetuned_failure_rate,baseline_mean_edits,finetuned_mean_edits,recovery_rate
0,cooking_query,14,1.000000,1.000000,1.214286,1.357143,0.000000
1,cleaning,8,1.000000,1.000000,5.125000,1.625000,0.000000
2,general_greet,17,1.000000,1.000000,2.058824,0.823529,0.000000
3,factoid,14,1.000000,1.000000,2.714286,1.285714,0.000000
4,hue_lightup,1,1.000000,1.000000,4.000000,1.000000,0.000000
5,podcasts,6,1.000000,1.000000,2.833333,0.833333,0.000000
6,post,2,1.000000,1.000000,3.500000,2.000000,0.000000
7,music_dislikeness,5,1.000000,1.000000,2.600000,1.400000,0.000000
8,greet,2,1.000000,1.000000,0.500000,0.000000,0.000000
9,hue_lightoff,9,1.000000,1.000000,3.444444,1.111111,0.000000


,recovery_status,samples,rate
0,recovered,3430,0.394797
1,always_correct,2950,0.339549
2,still_broken,1914,0.220304
3,regressed,394,0.045350


## Cell 13 — Build a canonical manifest for the report directory


In [13]:
canonical_manifest_rows = []
for path in sorted(REPORTS_DIR.glob('*')):
    if path.is_file():
        canonical_manifest_rows.append({
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "is_fixed_named": path.name.endswith('_fixed.csv') or path.name.endswith('_fixed.json') or path.name.endswith('_fixed.png'),
            "is_archive": False,
        })

canonical_manifest_df = pd.DataFrame(canonical_manifest_rows).sort_values("filename").reset_index(drop=True)
display(canonical_manifest_df.head(50))


,filename,size_bytes,is_fixed_named,is_archive
0,baseline_intent_probabilities.npy,3162560,False,False
1,baseline_predictions.csv,18767,False,False
2,calibration_ground_truth.png,67005,False,False
3,classification_report.json,12368,False,False
4,confidence_comparison.csv,310,False,False
5,confidence_distribution_comparison.png,79866,False,False
6,confusion_matrix.png,173408,False,False
7,dataset_report.md,3195,False,False
8,debugging_summary_07_08_09.json,1455,False,False
9,debugging_verdict_07_08_09.csv,358,False,False


## Cell 14 — Record the post-consolidation trust state


In [14]:
post_consolidation_verdict_df = pd.DataFrame([
    {
        "component": "07_upper_bound_intent_classifier",
        "status": "trusted",
        "reason": "Strong text-only upper bound with known minority-class limitations.",
    },
    {
        "component": "08_error_propagation_analysis",
        "status": "trusted",
        "reason": "Core downstream result is consistent: better ASR improves intent understanding.",
    },
    {
        "component": "10_results_diagnosis_debugging",
        "status": "trusted",
        "reason": "Valid forensic reasoning about which outputs were safe and unsafe.",
    },
    {
        "component": "11_fix_error_taxonomy",
        "status": "trusted_pending_result_review",
        "reason": "Produces corrected taxonomy outputs that are now promoted to canonical report files.",
    },
])

display(post_consolidation_verdict_df)


,component,status,reason
0,07_upper_bound_intent_classifier,trusted,Strong text-only upper bound with known minori...
1,08_error_propagation_analysis,trusted,Core downstream result is consistent: better A...
2,10_results_diagnosis_debugging,trusted,Valid forensic reasoning about which outputs w...
3,11_fix_error_taxonomy,trusted_pending_result_review,Produces corrected taxonomy outputs that are n...


## Cell 15 — Continuation roadmap

Now that the report state is clean, the project should move back into research rather than debugging.

The most valuable next questions are not simply about shaving a couple more WER points.
The stronger research questions are:

- Which ASR error types have the highest downstream impact?
- Are all WER reductions equally useful?
- Which intents are systematically fragile?
- Can we predict whether a transcription error will change intent?
- Is WER a good proxy for downstream understanding?
- Should ASR systems prioritize reducing high-impact errors rather than average errors?


In [15]:
continuation_plan_df = pd.DataFrame([
    {
        "next_notebook_or_task": "13_high_impact_error_analysis",
        "purpose": "Rank taxonomy categories by downstream failure rate and support.",
    },
    {
        "next_notebook_or_task": "14_is_wer_a_good_proxy",
        "purpose": "Test whether lower WER always predicts better downstream intent understanding.",
    },
    {
        "next_notebook_or_task": "15_predict_intent_change_from_asr_error",
        "purpose": "Model whether a transcript error will flip intent prediction.",
    },
    {
        "next_notebook_or_task": "16_final_research_summary",
        "purpose": "Assemble the final research narrative, claims, and figures.",
    },
])

display(continuation_plan_df)


,next_notebook_or_task,purpose
0,13_high_impact_error_analysis,Rank taxonomy categories by downstream failure...
1,14_is_wer_a_good_proxy,Test whether lower WER always predicts better ...
2,15_predict_intent_change_from_asr_error,Model whether a transcript error will flip int...
3,16_final_research_summary,"Assemble the final research narrative, claims,..."


## Cell 16 — Save consolidation artifacts


In [16]:
consolidation_summary = {
    "run_stamp": RUN_STAMP,
    "archive_dir": str(ARCHIVE_DIR),
    "broken_files_archived": archive_manifest_df.to_dict(orient="records"),
    "broken_files_deleted": delete_manifest_df.to_dict(orient="records"),
    "fixed_files_promoted": promotion_manifest_df.to_dict(orient="records"),
    "canonical_verification": verification_df.to_dict(orient="records"),
}

with open(REPORTS_DIR / "consolidation_summary_12.json", "w", encoding="utf-8") as f:
    json.dump(consolidation_summary, f, indent=2)

post_consolidation_verdict_df.to_csv(REPORTS_DIR / "post_consolidation_verdict.csv", index=False)
continuation_plan_df.to_csv(REPORTS_DIR / "research_continuation_plan.csv", index=False)
canonical_manifest_df.to_csv(REPORTS_DIR / "canonical_reports_manifest.csv", index=False)

print("Saved consolidation_summary_12.json")
print("Saved post_consolidation_verdict.csv")
print("Saved research_continuation_plan.csv")
print("Saved canonical_reports_manifest.csv")


Saved consolidation_summary_12.json
Saved post_consolidation_verdict.csv
Saved research_continuation_plan.csv
Saved canonical_reports_manifest.csv


## Cell 17 — Final summary


In [17]:
print("=" * 72)
print("VOXINTEL REPORT CONSOLIDATION COMPLETE")
print("=" * 72)
print(f"Archive directory: {ARCHIVE_DIR}")
print()
print("Promoted fixed files into canonical report names:")
print(promotion_manifest_df.to_string(index=False))
print()
print("Next recommended research direction:")
print(continuation_plan_df.to_string(index=False))


VOXINTEL REPORT CONSOLIDATION COMPLETE
Archive directory: c:\Users\ACER\OneDrive\Desktop\VoxIntel\reports\archive_broken_taxonomy_20260807_215056

Promoted fixed files into canonical report names:
                           source                 promoted_to status
 error_taxonomy_dataset_fixed.csv  error_taxonomy_dataset.csv copied
error_type_distribution_fixed.csv error_type_distribution.csv copied
    error_impact_matrix_fixed.csv     error_impact_matrix.csv copied
       intent_fragility_fixed.csv        intent_fragility.csv copied
        failure_gallery_fixed.csv         failure_gallery.csv copied
       recovery_summary_fixed.csv        recovery_summary.csv copied

Next recommended research direction:
                  next_notebook_or_task                                                                        purpose
          13_high_impact_error_analysis               Rank taxonomy categories by downstream failure rate and support.
                 14_is_wer_a_good_proxy Test